In [ ]:
import msprime, tskit
import numpy as np

In [ ]:

def span_definition(ts):
    node_spans = np.zeros(ts.num_nodes)
    for tree in ts.trees():
        for u in tree.nodes():
            node_spans[u] += tree.span
    return node_spans

def span(ts):
    num_children = np.zeros(ts.num_nodes, dtype=np.int32)
    span_start = np.zeros(ts.num_nodes)
    node_span = np.zeros(ts.num_nodes)
    
    for interval, edges_out, edges_in in ts.edge_diffs(include_terminal=True):
        for edge in edges_out:
            num_children[edge.parent] -= 1
            if num_children[edge.parent] == 0:
                node_span[edge.parent] += interval.left - span_start[edge.parent]
        for edge in edges_in:
            if num_children[edge.parent] == 0:
                span_start[edge.parent] = interval.left
            num_children[edge.parent] += 1
    # Set the sample spans afterwards, so internal samples are handled correctly
    node_span[ts.samples()] = ts.sequence_length
    return node_span


In [ ]:
def example4():        
    node_times = (0, 0, 0, 0, 1, 1, 3, 2, 2)
    samples = (0, 1, 2, 3)
    # (p, c, l, r)
    extended_edges = [
        (4, 0, 0, 10),
        (4, 1, 0, 5),
        (4, 1, 7, 10),
        (5, 2, 0, 2),
        (5, 2, 5, 10),
        (5, 3, 0, 10),
        (7, 2, 2, 5),
        (7, 4, 0, 10),
        (8, 1, 5, 7),
        (8, 5, 0, 10),
        (6, 7, 0, 10),
        (6, 8, 0, 10),
    ]
    edges = [
        (4, 0, 0, 10),
        (4, 1, 0, 5),
        (4, 1, 7, 10),
        (5, 2, 0, 2),
        (5, 2, 5, 10),
        (5, 3, 0, 2),
        (5, 3, 5, 10),
        (7, 2, 2, 5),
        (7, 4, 2, 5),
        (8, 1, 5, 7),
        (8, 5, 5, 7),
        (6, 3, 2, 5),
        (6, 4, 0, 2),
        (6, 4, 5, 10),
        (6, 5, 0, 2),
        (6, 5, 7, 10),
        (6, 7, 2, 5),
        (6, 8, 5, 7),
    ]
    tables = tskit.TableCollection(sequence_length=10)
    tables.sort()
    for n, t in enumerate(node_times):
        flags = tskit.NODE_IS_SAMPLE if n in samples else 0
        tables.nodes.add_row(time=t, flags=flags)
    for p, c, l, r in edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ts = tables.tree_sequence()
    tables.edges.clear()
    for p, c, l, r in extended_edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ets = tables.tree_sequence()
    assert ts.num_edges == 18
    assert ets.num_edges == 12
    return ts, ets

t4, et4 = example4()


In [ ]:

span1 = span_definition(t4)
print("[0.  1.  2.  3.  4.   5.  6.  7.  8.]")
print(span1)
span2 = span(t4)
print(span2)
span3 = span_definition(et4)
span4 = span(et4)
print()
print(span3)
print(span4)
print("correct for example 4")


In [ ]:
def example2():
        node_times = {
            0: 0,
            1: 0,
            2: 0,
            3: 0,
            4: 0,
            5: 0,
            6: 0,
            7: 0,
            8: 0,
            9: 0,
            10: 1,
            11: 2,
            12: 3,
            13: 4,
            14: 5,
            15: 6,
            16: 7,
            17: 8,
            18: 9,
            19: 10,
            20: 11,
            21: 12,
        }
        # (p,c,l,r)
        edges = [
            (10, 2, 0, 9),
            (10, 5, 0, 9),
            (11, 0, 0, 9),
            (11, 7, 0, 9),
            (12, 3, 3, 9),
            (12, 9, 3, 9),
            (13, 4, 0, 9),
            (13, 11, 0, 9),
            (14, 6, 0, 9),
            (14, 10, 0, 9),
            (15, 9, 0, 3),
            (15, 13, 0, 3),
            (16, 1, 0, 6),
            (16, 3, 0, 3),
            (16, 12, 3, 6),
            (17, 1, 6, 9),
            (17, 13, 6, 9),
            (18, 13, 3, 6),
            (18, 14, 0, 9),
            (18, 15, 0, 3),
            (18, 17, 6, 9),
            (19, 8, 0, 9),
            (19, 12, 6, 9),
            (19, 16, 0, 6),
            (20, 18, 0, 3),
            (20, 18, 6, 9),
            (20, 19, 0, 3),
            (20, 19, 6, 9),
            (21, 18, 3, 6),
            (21, 19, 3, 6),
        ]
        extended_edges = [
            (10, 2, 0.0, 9.0),
            (10, 5, 0.0, 9.0),
            (11, 0, 0.0, 9.0),
            (11, 7, 0.0, 9.0),
            (12, 3, 0.0, 9.0),
            (12, 9, 3.0, 9.0),
            (13, 4, 0.0, 9.0),
            (13, 11, 0.0, 9.0),
            (14, 6, 0.0, 9.0),
            (14, 10, 0.0, 9.0),
            (15, 9, 0.0, 3.0),
            (15, 13, 0.0, 9.0),
            (16, 1, 0.0, 6.0),
            (16, 12, 0.0, 9.0),
            (17, 1, 6.0, 9.0),
            (17, 15, 0.0, 9.0),
            (18, 14, 0.0, 9.0),
            (18, 17, 0.0, 9.0),
            (19, 8, 0.0, 9.0),
            (19, 16, 0.0, 9.0),
            (20, 18, 0.0, 3.0),
            (20, 18, 6.0, 9.0),
            (20, 19, 0.0, 3.0),
            (20, 19, 6.0, 9.0),
            (21, 18, 3.0, 6.0),
            (21, 19, 3.0, 6.0),
        ]
        samples = list(np.arange(10))
        tables = tskit.TableCollection(sequence_length=9)
        for (
            n,
            t,
        ) in node_times.items():
            flags = tskit.NODE_IS_SAMPLE if n in samples else 0
            tables.nodes.add_row(time=t, flags=flags)
        for p, c, l, r in edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ts = tables.tree_sequence()
        tables.edges.clear()
        for p, c, l, r in extended_edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ets = tables.tree_sequence()
        assert ts.num_edges == 30
        assert ets.num_edges == 26
        return ts, ets

t2, et2 = example2()


In [ ]:

span1 = span_definition(t2)
print("[0. 1. 2. 3. 4. 5. 6. 7. 8. 9. 10.11.12.13.14.15.16.17.18.19.20.21]")
print(span1)
span2 = span(t2)
print(span2)
span3 = span_definition(et2)
span4 = span(et2)
print()
print("[0. 1. 2. 3. 4. 5. 6. 7. 8. 9. 10.11.12.13.14.15.16.17.18.19.20.21]")
print(span3)
print(span4)
print("correct for example 2")

In [ ]:
def example1():
        node_times = {
            0: 0,
            1: 0,
            2: 0,
            3: 0,
            4: 1,
            5: 1,
            6: 4,
            7: 6,
            8: 10,
            9: 4,
            10: 12,
            11: 8,
            12: 8,
            13: 15,
        }
        # (p,c,l,r)
        edges = [
            (4, 0, 0, 9),
            (4, 1, 0, 9),
            (5, 2, 0, 6),
            (5, 3, 0, 9),
            (6, 4, 0, 3),
            (9, 5, 0, 3),
            (7, 4, 3, 6),
            (11, 7, 3, 6),
            (12, 5, 3, 6),
            (8, 2, 6, 9),
            (8, 4, 6, 9),
            (8, 6, 0, 3),
            (10, 5, 6, 9),
            (10, 8, 0, 3),
            (10, 8, 6, 9),
            (10, 9, 0, 3),
            (10, 11, 3, 6),
            (10, 12, 3, 6),
            (13, 10, 3, 6),
        ]
        extended_edges = [
            (4, 0, 0.0, 9.0),
            (4, 1, 0.0, 9.0),
            (5, 2, 0.0, 6.0),
            (5, 3, 0.0, 9.0),
            (6, 4, 0.0, 9.0),
            (9, 5, 0.0, 9.0),
            (7, 6, 0.0, 9.0),
            (11, 7, 0.0, 9.0),
            (12, 9, 0.0, 9.0),
            (8, 2, 6.0, 9.0),
            (8, 11, 0.0, 9.0),
            (10, 8, 0.0, 9.0),
            (10, 12, 0.0, 9.0),
            (13, 10, 3.0, 6.0),
        ]
        samples = list(np.arange(4))
        tables = tskit.TableCollection(sequence_length=9)
        for (
            n,
            t,
        ) in node_times.items():
            flags = tskit.NODE_IS_SAMPLE if n in samples else 0
            tables.nodes.add_row(time=t, flags=flags)
        for p, c, l, r in edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ts = tables.tree_sequence()
        tables.edges.clear()
        for p, c, l, r in extended_edges:
            tables.edges.add_row(parent=p, child=c, left=l, right=r)
        ets = tables.tree_sequence()
        assert ts.num_edges == 19
        assert ets.num_edges == 14
        return ts, ets

t1, et1 = example1()


In [ ]:
span1 = span_definition(t1)
span2 = span(t1)
print("[0. 1. 2. 3. 4. 5. 6. 7. 8. 9. 10.11.12.13.]")
print(span1)
print(span2)

span3 = span_definition(et1)
span4 = span(et1)
print()
print("[0. 1. 2. 3. 4. 5. 6. 7. 8. 9. 10.11.12.13.]")
print(span3)
print(span4)
print("correct for example 1")

In [ ]:
def attempt1():
    node_times = (0, 0, 0, 1, 2, 3)
    samples = (0, 1, 2)
    # (p, c, l, r)
    extended_edges = [
        (3, 0, 0, 4),
        (3, 1, 0, 3), 
        (4, 1, 3, 4),
        (4, 2, 0, 4), 
        (5, 3, 0, 4), 
        (5, 4, 0, 4),
    ]
    edges = [
        (3, 0, 0, 3),
        (3, 1, 0, 3), 
        (4, 1, 3, 4),
        (4, 2, 3, 4), 
        (5, 0, 3, 4),
        (5, 2, 0, 3), 
        (5, 3, 0, 3), 
        (5, 4, 3, 4),
    ]
    tables = tskit.TableCollection(sequence_length=4)
    tables.sort()
    for n, t in enumerate(node_times):
        flags = tskit.NODE_IS_SAMPLE if n in samples else 0
        tables.nodes.add_row(time=t, flags=flags)
    for p, c, l, r in edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ts = tables.tree_sequence()
    tables.edges.clear()
    for p, c, l, r in extended_edges:
        tables.edges.add_row(parent=p, child=c, left=l, right=r)
    ets = tables.tree_sequence()
    assert ts.num_edges == 8
    assert ets.num_edges == 6
    return ts, ets

t5, et5 = attempt1()


In [ ]:
span1 = span_definition(t5)
span2 = span(t5)
print("[0. 1. 2. 3. 4. 5.]")
print(span1)
print(span2)

span3 = span_definition(et5)
span4 = span(et5)
print()
print("[0. 1. 2. 3. 4. 5.]")
print(span3)
print(span4)
print("correct for attempt 1")